# Financial Data Analytics Capstone Project
### End-to-End Analysis Case Study: Data Cleaning, EDA, SQL & Power BI Preparation
**Tools Used**: Python (Pandas, Matplotlib, Seaborn), SQLite, Power BI, Excel
**Deliverables**: Cleaned Datasets, Visuals, Executive Report, SQL Queries, Power BI Guide
---


## 1. Business Objectives & Problem Statement
This capstone study examines the financial performance of global commercial activities across 5 countries (Canada, France, Germany, Mexico, USA) and 5 customer segments (Government, Small Business, Enterprise, Midmarket, Channel Partners).

Key questions addressed:
- What are the overall revenue, cost, and profitability metrics?
- Which countries and customer segments contribute the most to the bottom line?
- Why is the Enterprise segment exhibiting negative profitability?
- What is the effect of discount bands on margin erosion?
- What seasonal patterns exist across 2013-2014?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('Environment and libraries ready!')

## 2. Data Ingestion & Initial Audit

In [ ]:
# Load raw dataset
df_raw = pd.read_excel('sheet.xlsx')
print(f'Rows: {df_raw.shape[0]}, Columns: {df_raw.shape[1]}')
print('Raw columns:', df_raw.columns.tolist())
df_raw.head()

## 3. Data Cleaning & Feature Engineering
- Trim leading/trailing whitespace in column names (e.g. ' Sales' -> 'Sales').
- Impute missing values in 'Discount Band' with 'None'.
- Ensure numeric casting for financial values.
- Parse Dates and compute Profit Margin %.

In [ ]:
df = df_raw.copy()
df.columns = [c.strip() for c in df.columns]
df['Discount Band'] = df['Discount Band'].fillna('None')

numeric_fields = ['Units Sold', 'Manufacturing Price', 'Sale Price', 'Gross Sales', 'Discounts', 'Sales', 'COGS', 'Profit']
for col in numeric_fields:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month Number'] = df['Date'].dt.month
df['Month Name'] = df['Date'].dt.strftime('%B')
df['Quarter'] = df['Date'].dt.to_period('Q').astype(str)
df['Profit Margin %'] = (df['Profit'] / df['Sales']) * 100

print('Missing Values Count:')
print(df.isnull().sum())
df.describe()

## 4. Key Performance Indicators (KPIs)

In [ ]:
total_gross = df['Gross Sales'].sum()
total_disc = df['Discounts'].sum()
total_net = df['Sales'].sum()
total_cogs = df['COGS'].sum()
total_profit = df['Profit'].sum()
total_units = df['Units Sold'].sum()
margin_pct = (total_profit / total_net) * 100
discount_rate = (total_disc / total_gross) * 100

kpi_df = pd.DataFrame({
    'KPI Metric': ['Total Gross Sales', 'Total Discounts', 'Total Net Sales', 'Total COGS', 'Total Net Profit', 'Units Sold', 'Overall Margin %', 'Discount Rate %'],
    'Value': [
        f'${total_gross:,.2f}',
        f'${total_disc:,.2f}',
        f'${total_net:,.2f}',
        f'${total_cogs:,.2f}',
        f'${total_profit:,.2f}',
        f'{total_units:,.0f}',
        f'{margin_pct:.2f}%',
        f'{discount_rate:.2f}%'
    ]
})
kpi_df

## 5. Segment Performance & The Enterprise Loss Discovery

In [ ]:
seg_summary = df.groupby('Segment').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Units Sold': 'sum'
}).sort_values(by='Sales', ascending=False)
seg_summary['Profit Margin %'] = (seg_summary['Profit'] / seg_summary['Sales']) * 100
seg_summary['Sales Share %'] = (seg_summary['Sales'] / total_net) * 100
seg_summary

## 6. Country & Geographic Distribution

In [ ]:
country_summary = df.groupby('Country').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Units Sold': 'sum'
}).sort_values(by='Profit', ascending=False)
country_summary['Profit Margin %'] = (country_summary['Profit'] / country_summary['Sales']) * 100
country_summary

## 7. Discount Band Sensitivity Analysis

In [ ]:
disc_summary = df.groupby('Discount Band').agg({
    'Gross Sales': 'sum',
    'Discounts': 'sum',
    'Sales': 'sum',
    'Profit': 'sum'
}).reindex(['None', 'Low', 'Medium', 'High'])
disc_summary['Profit Margin %'] = (disc_summary['Profit'] / disc_summary['Sales']) * 100
disc_summary

## 8. Export Cleaned Dataset for Power BI & SQL Database Sync

In [ ]:
df.to_csv('Financial_Data_Cleaned.csv', index=False)
df.to_excel('Financial_Data_Cleaned.xlsx', index=False)

conn = sqlite3.connect('financial_analytics.db')
df.to_sql('financials', conn, if_exists='replace', index=False)
conn.close()
print('Cleaned files exported and SQLite database updated!')